## Import libraries

In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import tifffile 
from careamics import CAREamist
from careamics.config import create_care_configuration
from careamics.utils.metrics import scale_invariant_psnr


## Load the data

In [ ]:
data_split = "U2OS_splits_n100_seed42"
prepared_dir = Path(f"../data_careamics/{data_split}")

train_path = prepared_dir / "train" / "low"
train_target_path = prepared_dir / "train" / "high"
val_path = prepared_dir / "validation" / "low"
val_target_path = prepared_dir / "validation" / "high"

test_path = prepared_dir / "test" / "low"
test_target_path = prepared_dir / "test" / "high"

## Configure the training parameters

In [ ]:
# create a configuration

batch_s = 512
num_ep = 30
num_steps = 20
patch_s = 128
data_t = "tiff"
training_axes = "YX"



config = create_care_configuration(
    experiment_name="care_U20S",
    data_type=data_t,
    axes=training_axes,
    patch_size=(patch_s, patch_s),
    batch_size=batch_s,
    num_epochs=num_ep,
    num_steps=num_steps,
)

print(config)

### Train

A `CAREamist` can be created using a configuration alone, and then be trained by using
the data already loaded in memory. Since we do not provide validation data explicitly,
it will be split automatically (see configuration `n_val_patches` parameter).

In [ ]:
# instantiate a CAREamist
careamist = CAREamist(source=config)


# train
careamist.train(
    train_source=train_path,
    train_target=train_target_path,
    val_source=val_path,
    val_target=val_target_path
)

## Dsiplay learning process.

In [ ]:
loss_dict = careamist.get_losses()
plt.plot(loss_dict["train_epoch"], loss_dict["train_loss"], loss_dict["val_epoch"], loss_dict["val_loss"])
plt.legend(["Train loss", "Val loss"])
plt.title("Losses")

## Predict with CAREamics

Prediction is done with the same `CAREamist` used for training, and we can use the test
set.

In [ ]:
prediction = careamist.predict(source=test_path)

## Visualize the prediction

In [ ]:
# Show two images
# test_GT_path = root_path / "test" / "GT"
test_GT_files = list(test_target_path.rglob("*.tif"))
test_GT_files.sort()

# test_low_path = root_path / "test" / "low"
test_low_files = list(test_path.rglob("*.tif"))
test_low_files.sort()

test_GT = [tifffile.imread(f) for f in test_GT_files]
test_low = [tifffile.imread(f) for f in test_low_files]

# images to show
images = np.random.choice(range(len(test_GT)), 3)

fig, ax = plt.subplots(3, 3, figsize=(15, 17))
fig.tight_layout()

for i in range(3):
    pred_image = prediction[images[i]].squeeze()

    psnr_noisy = scale_invariant_psnr(test_GT[images[i]], test_low[images[i]])
    psnr_result = scale_invariant_psnr(test_GT[images[i]], pred_image)

    ax[i, 0].imshow(test_low[images[i]], cmap="gray")
    ax[i, 0].title.set_text(f"Noisy\nPSNR: {psnr_noisy:.2f}")

    ax[i, 1].imshow(pred_image, cmap="gray")
    ax[i, 1].title.set_text(f"Prediction\nPSNR: {psnr_result:.2f}")

    ax[i, 2].imshow(test_GT[images[i]], cmap="gray")
    ax[i, 2].title.set_text("Ground-truth")

### Compute metrics

In [ ]:
psnrs = np.zeros((len(prediction), 1))

for i, (pred, gt) in enumerate(zip(prediction, test_GT)):
    psnrs[i] = scale_invariant_psnr(gt, pred.squeeze())

print(f"PSNR: {psnrs.mean():.2f} +/- {psnrs.std():.2f}")